# 🧪 Étape 6 : Évaluation Métrique & Robustesse

Cette étape correspond au sixième chapitre du cours. L'objectif est de mettre en place un **protocole d'évaluation rigoureux** et de calculer les métriques clés de performance pour valider scientifiquement la qualité de notre modèle de prédiction des prix immobiliers.

Nous appliquons :
- Un **calcul détaillé des métriques d'erreur** (MAE, RMSE, R²)
- Une **validation croisée K-Fold** pour mesurer la robustesse du modèle

## 1. Préparation de l'environnement

Importation des bibliothèques d'évaluation et chargement du modèle entraîné lors de l'étape 5.

In [2]:
import os
import sys
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Chargement du dataset
df = pd.read_csv("../data/processed/cleaned_data_sample.csv")

# Chargement du modèle RandomForest entraîné à l'étape 5
model = joblib.load("../models/rf_house_prices.pkl")

print(f"✔ Dataset chargé : {df.shape}")
print(f"✔ Modèle RandomForest chargé : {type(model).__name__}")

✔ Dataset chargé : (1460, 85)
✔ Modèle RandomForest chargé : RandomForestRegressor


## 2. Évaluation du modèle Tabulaire (RandomForest)

Calcul des trois métriques de référence pour la régression :
- **MAE** (Mean Absolute Error) — erreur moyenne en dollars
- **RMSE** (Root Mean Squared Error) — pénalise davantage les grosses erreurs
- **R²** (Coefficient de détermination) — proportion de la variance expliquée par le modèle

In [3]:
# Préparation des données (mêmes features qu'à l'étape 5)
features = [
    "GrLivArea",
    "OverallQual",
    "HouseAge",
    "GarageCars",
    "TotalBsmtSF",
    "coef_multiplicateur"
]
target = "SalePrice"

X = df[features]
y = df[target]

# Split identique à l'étape 5 (random_state=42 pour reproductibilité)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Prédictions sur le test set
y_pred = model.predict(X_test)

# Calcul des métriques
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 ÉVALUATION SUR LE TEST SET")
print(f"MAE  : {mae:,.2f} $")
print(f"RMSE : {rmse:,.2f} $")
print(f"R²   : {r2:.4f}")

📊 ÉVALUATION SUR LE TEST SET
MAE  : 20,700.46 $
RMSE : 35,057.79 $
R²   : 0.7334


## 3. Validation croisée (K-Fold)

Pour évaluer la **robustesse** du modèle au-delà d'un simple split train/test, nous appliquons une **validation croisée K-Fold** avec K=5.

**Principe :** le dataset est divisé en 5 sous-ensembles. Le modèle est entraîné et évalué 5 fois, en utilisant à chaque fois un sous-ensemble différent comme test. La moyenne des scores donne une estimation plus fiable de la performance réelle.

> **Pourquoi K-Fold classique et non TimeSeriesSplit ?** Les données immobilières ne présentent pas de structure temporelle stricte (le prix d'une maison ne dépend pas de l'ordre dans lequel les ventes sont enregistrées dans le dataset). Une validation croisée classique est donc adaptée.

In [4]:
# Validation croisée 5-Fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Scores R² sur chaque fold
cv_r2 = cross_val_score(model, X, y, cv=kfold, scoring="r2")

# Scores MAE (négatif par convention scikit-learn, on inverse pour lisibilité)
cv_mae = -cross_val_score(model, X, y, cv=kfold, scoring="neg_mean_absolute_error")

print("📈 VALIDATION CROISÉE 5-FOLD\n")

print("Scores R² par fold :")
for i, score in enumerate(cv_r2, 1):
    print(f"  Fold {i} : {score:.4f}")

print(f"\n  R² moyen   : {cv_r2.mean():.4f}")
print(f"  R² écart-type : {cv_r2.std():.4f}")

print(f"\n  MAE moyen  : {cv_mae.mean():,.2f} $")
print(f"  MAE écart-type : {cv_mae.std():,.2f} $")

📈 VALIDATION CROISÉE 5-FOLD

Scores R² par fold :
  Fold 1 : 0.7319
  Fold 2 : 0.7416
  Fold 3 : 0.7337
  Fold 4 : 0.7363
  Fold 5 : 0.8115

  R² moyen   : 0.7510
  R² écart-type : 0.0304

  MAE moyen  : 21,352.49 $
  MAE écart-type : 1,493.29 $


## ✅ Conclusion

Le modèle RandomForest démontre une performance **robuste et reproductible** :
- **R² test set : 0.73** (mesure simple)
- **R² 5-Fold : 0.75 ± 0.03** (validation croisée)
- **Erreur moyenne (MAE) : ~21 000 $**

La très faible variance entre les folds (écart-type R² de 0.03) confirme que le modèle ne souffre pas de surapprentissage sur un sous-ensemble particulier des données. Les performances sont cohérentes quelle que soit la partition du dataset, ce qui valide scientifiquement l'utilisation de ce modèle pour la prédiction des prix immobiliers.